# Judge saved validation predictions

Select the existing **Qwen Commit Review (ROCm)** kernel. This notebook runs the pinned Qwen3-4B-Instruct-2507 judge locally against saved Vast predictions. Public weights download into the local Hugging Face cache; diffs and predictions stay local. It reads only the approved validation cohort, including formatting failures. Model judgments are proposals for human review, not verified correctness.

In [ ]:
import sys
from pathlib import Path

project_directory = Path.cwd()
if project_directory.name == "notebooks":
    project_directory = project_directory.parent
sys.path.insert(0, str(project_directory / "experiments/cuda"))

In [ ]:
from judge import LocalJudge, approved_predictions, review_interactively, run_judging

## Select the saved run

Replace `RUN_ID` with the completed validation run directory. It must contain `validation.json`, `validation-input.parquet`, `validation-references.parquet`, and `validation-predictions.parquet`. The shared evaluator verifies their exact join before loading the judge. No prepared test files are opened.

In [ ]:
run_directory = project_directory / "experiments/cuda/artifacts/vast/RUN_ID"
rows, predictions_sha256 = approved_predictions(run_directory)
{"approved_validation_rows": len(rows), "predictions_sha256": predictions_sha256}

## Generate judgment proposals

The judge sees the full diff and raw candidate, without reference wording or generator identity. Prompts over 8,192 tokens are recorded as uncertain for manual review, never truncated. Greedy generation allows 512 output tokens; invalid or unfinished responses also require review. Each row is saved immediately to `correctness-judge.json`. Rerunning resumes with the same model revision, prompt, settings, and runtime. Add `limit=3` to `run_judging` for a short first run.

In [ ]:
grader = LocalJudge()

In [ ]:
summary = run_judging(run_directory, grader)
summary

## Review each verdict

This step needs no GPU; when returning just to review, run the imports and run-selection cells, then this cell. Inspect each full diff with `d`, choose `a` (accurate), `p` (partial), `i` (incorrect), or `u` (uncertain), and give an evidence rationale. Use `q` to stop and resume later. Uncertain decisions are revisited in the next session. References provide context, not an exact-match target.

Decisions save separately to `correctness-human-review.json`. `correctness-summary.json` reports model and human grades separately. Correctness is accurate subjects divided by the entire approved cohort; usable output additionally requires a formatting pass. Rates stay unavailable while that grader has pending or uncertain rows. The cohort remains fixed, and this is not accuracy over the full validation or test split. Keep all private artifacts and notebook outputs out of Git.

In [ ]:
review_interactively(run_directory)